<a href="https://colab.research.google.com/github/Suim00/hdac-team5-knhanes/blob/main/11_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 11주차 DAG와 복합표본 회귀 (9팀)

국민건강영양조사 2024년. 연구질문: 우리나라 성인에서 아침 결식은 아침 식사를 하는 경우에 비해 비만(BMI 25 이상) 유병과 연관되는가?

- 추정 대상: 총효과(관련성). 모형 1 = 노출만, 모형 2 = DAG의 최소 보정집합. 로지스틱 회귀(오즈비).
- 구성: 1~2 10주차 코드(불러오기, 변수 만들기, `in_study`) → 3 보정변수와 완전 사례 → 4 설계 기반 로지스틱 회귀 → 5 Table 2

## 1. Drive 연결과 정의표의 변수만 읽기 (10주차와 같은 코드)

`[11주차 추가]`가 붙은 줄만 10주차와 다르다. 소득·신체활동·체중조절 노력 변수를 더 읽는다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/hdac_data/HN24_ALL(SAS)'
DATA_FILE = f'{DATA_DIR}/hn24_all.sas7bdat'

In [ ]:
!pip -q install pyreadstat
import pyreadstat, numpy as np, pandas as pd

need = ['age', 'sex', 'HE_prg', 'L_BR_FQ', 'HE_ht', 'HE_wt', 'HE_BMI', 'HE_obe',
        'wt_tot', 'kstrata', 'psu', 'town_t', 'educ', 'sm_presnt', 'dr_month', 'N_DIET', 'N_EN',
        'ho_incm', 'pa_aerobic', 'BO2_1']   # [11주차 추가] 가구소득 4분위, 유산소 신체활동, 1년간 체중조절 여부

_, meta0 = pyreadstat.read_sas7bdat(DATA_FILE, metadataonly=True)
lower_map = {c.lower(): c for c in meta0.column_names}
present = {v: lower_map.get(v.lower()) for v in need}
assert all(present.values()), [v for v, a in present.items() if a is None]

df, meta = pyreadstat.read_sas7bdat(DATA_FILE, usecols=list(present.values()))
df = df.rename(columns={a: v for v, a in present.items()})
print(df.shape)

## 2. 노출·결과·공변량 만들기와 `in_study` (10주차와 같은 코드)

In [ ]:
df['breakfast_skip'] = df['L_BR_FQ'].map({1: 0, 2: 0, 3: 1, 4: 1})
df['obese'] = np.where(df['HE_obe'].isna(), np.nan, (df['HE_obe'] >= 4).astype(float))
df['educ_c'] = df['educ'].where(~df['educ'].isin([88, 99]))
df['edu4'] = df['educ_c'].map({1: 1, 2: 1, 3: 1, 4: 2, 5: 3, 6: 4, 7: 4, 8: 4})   # 묶음은 팀이 확정
df['N_DIET_c'] = df['N_DIET'].where(~df['N_DIET'].isin([9]))
df['clu'] = df['kstrata'].astype(str) + '_' + df['psu'].astype(str)   # 집락 식별자 (층 + 조사구)

In [ ]:
conds = [
    ('1. 만 19세 이상', df['age'] >= 19, '연구 대상 아님(비해당)'),
    ('2. 임신 중 아님', df['HE_prg'] != 1, '제외 기준(임신)'),
    ('3. 아침식사 빈도 응답', df['L_BR_FQ'].isin([1, 2, 3, 4]), '결측(노출 정의 불가)'),
    ('4. 신장·체중·BMI 값 있음', df[['HE_ht', 'HE_wt', 'HE_BMI']].notna().all(axis=1),
     '결측(BMI 이상값 기준은 적용하지 않음)'),
    ('5. 가중치·층·집락 유효',
     df['wt_tot'].notna() & (df['wt_tot'] > 0) & df['kstrata'].notna() & df['psu'].notna(),
     '설계상 제외'),
]

cur = pd.Series(True, index=df.index)
rows = [{'단계': '0. 전체 조사 참여자', '제외 인원': 0, '남은 인원': int(cur.sum()), '사유': '출발 대상'}]
for name, cond, why in conds:
    before = int(cur.sum())
    cur = cur & cond
    rows.append({'단계': name, '제외 인원': before - int(cur.sum()), '남은 인원': int(cur.sum()), '사유': why})

df['in_study'] = cur.astype(int)      # 분석 대상 표시변수
df['design_ok'] = conds[-1][1]        # 설계 변수 유효 표시변수
flow = pd.DataFrame(rows)
print(flow.to_string(index=False))

base = (df['age'] >= 19) & (df['HE_prg'] != 1) & df['L_BR_FQ'].isin([1, 2, 3, 4])
print()
print('4단계 변수별 결측:', df.loc[base, ['HE_ht', 'HE_wt', 'HE_BMI']].isna().sum().to_dict())

In [ ]:
d = df[df['design_ok']].copy()        # 설계 정보가 있는 전체 (도메인 분석용)
print('설계 자료 n:', len(d), '/ 분석 대상(도메인) n:', int(d['in_study'].sum()))
print('층 수:', d['kstrata'].nunique(), '/ 집락(PSU) 수:', d['clu'].nunique())
psu_per_strata = d.groupby('kstrata')['clu'].nunique()
print('층당 PSU 수 최소:', int(psu_per_strata.min()), '(2 이상이어야 분산 추정 가능)')
print('가중치 합(추정 모집단, 명):', round(d['wt_tot'].sum()))

In [ ]:
def _var_ratio(d, u):
    t = pd.DataFrame({'h': d['kstrata'].values, 'c': d['clu'].values, 'u': u})
    uc = t.groupby(['h', 'c'])['u'].sum().reset_index()
    var = 0.0
    for h, g in uc.groupby('h'):
        n = len(g)
        if n > 1:
            var += n / (n - 1) * ((g['u'] - g['u'].mean()) ** 2).sum()
    return var

def svy_mean(d, y, dom):
    """도메인(dom=True)에서 y의 가중 평균과 SE. y가 결측인 행은 도메인에서 제외."""
    ok = (dom & d[y].notna()).values
    w = d['wt_tot'].values
    yy = d[y].fillna(0).values.astype(float)
    W = (w * ok).sum()
    m = (w * ok * yy).sum() / W
    u = w * ok * (yy - m) / W
    return m, np.sqrt(_var_ratio(d, u)), int(ok.sum())

## 3. 보정변수와 완전 사례

보정집합(DAG 최소 보정집합, 총효과): 나이, 성별, 교육, 소득, 거주지, 흡연, 음주, 신체활동.
에너지 섭취(`N_EN`, 매개 가능)와 식사요법(`N_DIET`, 비만의 결과일 수 있음)은 넣지 않는다.

In [ ]:
# 새로 쓰는 보정변수의 원 코드 빈도 (분석 대상 안, 결측 포함)
ins = d['in_study'] == 1
print('ho_incm 가구 소득 4분위 (1 하, 2 중하, 3 중상, 4 상; 이용지침서 93쪽)')
print(d.loc[ins, 'ho_incm'].value_counts(dropna=False).sort_index().to_string())
print('\npa_aerobic 유산소 신체활동 실천 (0 미실천, 1 실천; 이용지침서 169쪽)')
print(d.loc[ins, 'pa_aerobic'].value_counts(dropna=False).sort_index().to_string())
assert set(d.loc[ins, 'ho_incm'].dropna().unique()) <= {1, 2, 3, 4}
assert set(d.loc[ins, 'pa_aerobic'].dropna().unique()) <= {0, 1}
for v in ['sm_presnt', 'dr_month', 'town_t', 'sex']:
    print(f'\n{v}'); print(d.loc[ins, v].value_counts(dropna=False).sort_index().to_string())

In [ ]:
# 보정변수 (모형 2) 와 완전 사례 표시
ADJ = ['age', 'sex', 'edu4', 'ho_incm', 'town_t', 'sm_presnt', 'dr_month', 'pa_aerobic']
d['complete_case'] = (ins & d[ADJ].notna().all(axis=1)).astype(int)

n_in = int(ins.sum())
n_cc = int(d['complete_case'].sum())
print('Table 1 분석 대상(in_study = 1):', n_in)
print('보정변수가 모두 있는 사람(complete_case = 1):', n_cc)
print('차이(회귀에서 빠지는 인원):', n_in - n_cc)
print('\n보정변수별 결측 인원 (분석 대상 안)')
print(d.loc[ins, ADJ].isna().sum().to_string())
print('\n순서대로 빼 갈 때 남는 인원')
cur_ = ins.copy()
for v in ADJ:
    before = int(cur_.sum()); cur_ = cur_ & d[v].notna()
    print(f'  {v:10s} 결측으로 {before - int(cur_.sum()):4d}명 제외 -> {int(cur_.sum())}명')

In [ ]:
# 완전 사례로 남은 사람과 빠진 사람이 노출·결과에서 얼마나 다른지 (비가중 집계)
g = d[ins].copy()
g['집단'] = np.where(g['complete_case'] == 1, '완전 사례', '결측으로 제외')
print(g.groupby('집단').agg(n=('obese', 'size'),
                            결식_비율=('breakfast_skip', 'mean'),
                            비만_비율=('obese', 'mean')).round(3).to_string())

## 4. 설계 기반 로지스틱 회귀

- 추정: 가중치 `wt_tot`로 의사우도(pseudo-likelihood)를 최대화한다.
- 분산: 층 `kstrata`, 집락 `clu`(층+조사구)를 반영한 Taylor 선형화(sandwich) 분산. 자유도 = 집락 수 − 층 수 (t 분포로 95% 신뢰구간과 p 값).
- 도메인: 분석 대상 밖(완전 사례가 아닌 행)은 행을 지우지 않고 가중치를 0으로 둔다.
- 모형 1과 모형 2는 같은 사람(`complete_case = 1`)으로 적합한다.

In [ ]:
from scipy import stats

def svy_logit(d, y, xcols, dom, tol=1e-9, maxit=100):
    """설계 기반 로지스틱 회귀. dom=True 인 행만 쓰고 나머지는 가중치 0으로 남긴다."""
    ok = np.asarray(dom) & d[y].notna().values & d[xcols].notna().all(axis=1).values
    X = np.column_stack([np.ones(len(d)), d[xcols].fillna(0).values.astype(float)])
    yy = d[y].fillna(0).values.astype(float)
    w = d['wt_tot'].values * ok
    beta = np.zeros(X.shape[1])
    for it in range(maxit):
        p = 1 / (1 + np.exp(-X @ beta))
        A = X.T @ (X * (w * p * (1 - p))[:, None])
        step = np.linalg.solve(A, X.T @ (w * (yy - p)))
        beta = beta + step
        if np.max(np.abs(step)) < tol:
            break
    else:
        raise RuntimeError('수렴하지 않음')
    p = 1 / (1 + np.exp(-X @ beta))
    A = X.T @ (X * (w * p * (1 - p))[:, None])
    u = X * (w * (yy - p))[:, None]
    t = pd.DataFrame(u)
    t['h'] = d['kstrata'].values
    t['c'] = d['clu'].values
    uc = t.groupby(['h', 'c']).sum()
    B = np.zeros((X.shape[1], X.shape[1]))
    for h, g in uc.groupby(level=0):
        n = len(g)
        if n > 1:
            z = g.values - g.values.mean(axis=0)
            B += n / (n - 1) * (z.T @ z)
    Ai = np.linalg.inv(A)
    V = Ai @ B @ Ai
    return {'names': ['const'] + list(xcols), 'beta': beta, 'se': np.sqrt(np.diag(V)),
            'df': d['clu'].nunique() - d['kstrata'].nunique(), 'n': int(ok.sum()),
            'X': X, 'y': yy, 'w': w, 'ok': ok}

def or_row(fit, name):
    i = fit['names'].index(name)
    b, se, df_ = fit['beta'][i], fit['se'][i], fit['df']
    tc = stats.t.ppf(0.975, df_)
    return {'OR': np.exp(b), 'CI_low': np.exp(b - tc * se), 'CI_high': np.exp(b + tc * se),
            'p': 2 * (1 - stats.t.cdf(abs(b / se), df_)), 'beta': b, 'SE(beta)': se}

In [ ]:
# 모형에 쓰는 변수 만들기 (결측 행은 완전 사례가 아니므로 가중치 0)
d['age10'] = (d['age'] - 50) / 10          # 10세 단위, 50세 중심
d['age10_sq'] = d['age10'] ** 2            # 나이와 비만 유병의 관계가 직선이 아닐 수 있어 제곱항 포함
d['female'] = (d['sex'] == 2).astype(float)
for k in [2, 3, 4]:
    d[f'edu_{k}'] = (d['edu4'] == k).astype(float)     # 기준: 1 초등학교 이하
    d[f'inc_{k}'] = (d['ho_incm'] == k).astype(float)  # 기준: 1 하
d['rural'] = (d['town_t'] == 2).astype(float)          # 기준: 동

X1 = ['breakfast_skip']
X2 = X1 + ['age10', 'age10_sq', 'female', 'edu_2', 'edu_3', 'edu_4',
           'inc_2', 'inc_3', 'inc_4', 'rural', 'sm_presnt', 'dr_month', 'pa_aerobic']
dom = (d['complete_case'] == 1).values

fit1 = svy_logit(d, 'obese', X1, dom)
fit2 = svy_logit(d, 'obese', X2, dom)
print('모형 1 n =', fit1['n'], '/ 모형 2 n =', fit2['n'], '/ 같은 사람인지:', bool((fit1['ok'] == fit2['ok']).all()))
print('자유도(집락 수 - 층 수):', fit1['df'])
r1, r2 = or_row(fit1, 'breakfast_skip'), or_row(fit2, 'breakfast_skip')
print('모형 1 OR %.3f (%.3f-%.3f) p=%.4f' % (r1['OR'], r1['CI_low'], r1['CI_high'], r1['p']))
print('모형 2 OR %.3f (%.3f-%.3f) p=%.4f' % (r2['OR'], r2['CI_low'], r2['CI_high'], r2['p']))

In [ ]:
# 점검 1: 점 추정치가 일반 로지스틱(빈도 가중)과 같은지 (SE는 설계 기반이라 다름)
import statsmodels.api as sm
for nm, f in [('모형 1', fit1), ('모형 2', fit2)]:
    m = sm.GLM(f['y'][f['ok']], f['X'][f['ok']], family=sm.families.Binomial(),
               freq_weights=f['w'][f['ok']]).fit()
    print(nm, '계수 최대 차이:', float(np.max(np.abs(m.params - f['beta']))))

# 점검 2: 노출 계수의 설계 기반 SE와, 층·집락을 무시한 SE 비교 (설계효과 방향 확인)
m2 = sm.GLM(fit2['y'][fit2['ok']], fit2['X'][fit2['ok']], family=sm.families.Binomial(),
            freq_weights=fit2['w'][fit2['ok']]).fit()
i = fit2['names'].index('breakfast_skip')
print('모형 2 노출 SE: 설계 기반 %.4f / 단순 가중(참고용) %.4f' % (fit2['se'][i], m2.bse[i]))

In [ ]:
# 참고: 모형 1을 Table 1 분석 대상 전체(5,749명)로 적합하면 어떻게 달라지는가 (Table 2에는 넣지 않음)
fit1_all = svy_logit(d, 'obese', X1, (d['in_study'] == 1).values)
ra = or_row(fit1_all, 'breakfast_skip')
print('모형 1 (분석 대상 전체) n = %d, OR %.3f (%.3f-%.3f)' % (fit1_all['n'], ra['OR'], ra['CI_low'], ra['CI_high']))

# 참고: 완전 사례 안의 가중 비만 유병률 (오즈비와 유병률비의 차이를 해석할 때 참고)
dom_s = d['complete_case'] == 1
for name, cond in [('결식군', d['breakfast_skip'] == 1), ('비결식군', d['breakfast_skip'] == 0)]:
    m, se, n = svy_mean(d, 'obese', dom_s & cond)
    print(f'{name}: 가중 비만 유병률 {100 * m:.1f}% (SE {100 * se:.2f}), n = {n}')
m, se, n = svy_mean(d, 'obese', dom_s)
print(f'완전 사례 전체: 가중 비만 유병률 {100 * m:.1f}% (SE {100 * se:.2f}), n = {n}')

## 5. Table 2 형식으로 정리

In [ ]:
def fmt_p(p):
    return '<0.001' if p < 0.001 else f'{p:.3f}'

rows = []
for nm, f, r, xs in [('모형 1 (무보정)', fit1, r1, X1), ('모형 2 (보정)', fit2, r2, X2)]:
    rows.append({'모형': nm, '노출': '결식군 (기준: 비결식군)',
                 'OR': f"{r['OR']:.2f}", '95% CI': f"{r['CI_low']:.2f}–{r['CI_high']:.2f}", 'p': fmt_p(r['p']),
                 '분석 인원': f"{f['n']:,}", 'Table 1 대비 차이': f"{n_in - f['n']:,}"})
table2 = pd.DataFrame(rows)
print(table2.to_string(index=False))

# md에 붙여 넣을 형식
print()
print('| ' + ' | '.join(table2.columns) + ' |')
print('|' + '---|' * len(table2.columns))
for _, r in table2.iterrows():
    print('| ' + ' | '.join(str(v) for v in r) + ' |')

In [ ]:
# 모형 2의 보정변수 오즈비 (표에는 넣지 않아도 됨. 부호가 상식과 맞는지 확인용)
for nm in fit2['names'][1:]:
    r = or_row(fit2, nm)
    print(f"{nm:12s} OR {r['OR']:.2f} ({r['CI_low']:.2f}-{r['CI_high']:.2f})")

In [ ]:
# 12주차 민감도분석 대비: 체중 조절 노력자 규모 (BO2_1: 1 감소, 2 유지, 3 증가, 4 노력 안 함; 8·9 결측 처리)
d['BO2_1_c'] = d['BO2_1'].where(d['BO2_1'].isin([1, 2, 3, 4]))
cc = d['complete_case'] == 1
print('완전 사례 안 BO2_1 분포 (결측 포함)')
print(d.loc[cc, 'BO2_1_c'].value_counts(dropna=False).sort_index().to_string())
lose = cc & (d['BO2_1_c'] == 1)
print('\n체중 감소 노력자 수:', int(lose.sum()), '/ 이들을 뺀 완전 사례 수:', int((cc & ~lose).sum()))
print('체중 감소 노력자 중 결식군 비율(비가중): %.3f / 그 외: %.3f' % (
    d.loc[lose, 'breakfast_skip'].mean(), d.loc[cc & ~lose, 'breakfast_skip'].mean()))

In [ ]:
# 최종 숫자 대조
assert fit1['n'] == fit2['n'] == n_cc
print('Table 1 분석 대상', n_in, '-> 완전 사례', n_cc, '(차이', n_in - n_cc, ')')
print('모형 1·2 분석 인원:', fit1['n'], fit2['n'])